# Lab 03: Deploy CertAgent to Amazon Bedrock AgentCore Runtime

## Overview

Deploy a persistent AI agent to AgentCore Runtime using the `bedrock-agentcore-starter-toolkit` SDK.

```
User --> AgentCore Runtime (CertAgent) --> Lambda tools --> DynamoDB/Secrets Manager
```

The toolkit handles: Docker build (CodeBuild ARM64), ECR push, IAM roles, and deployment.

**Estimated time:** 30 minutes

**Important:** Run cells in order. Do NOT restart kernel between configure/launch/invoke.

## Step 1 - Install dependencies

In [ ]:
!pip install -q bedrock-agentcore-starter-toolkit strands-agents strands-agents-bedrock boto3

## Step 2 - Load workshop configuration

In [ ]:
import json, pathlib, boto3, os
from boto3.session import Session

config = json.loads(pathlib.Path('/tmp/certagent_config.json').read_text())
globals().update(config)
REPO_DIR = pathlib.Path(REPO_DIR)

boto_session = Session(region_name=AWS_REGION)
region = AWS_REGION

print(f'Region: {AWS_REGION}')
print(f'Scan Lambda: {LAMBDA_SCAN}')
print('Config loaded')

## Step 3 - Write the CertAgent server code

This is the agent entrypoint deployed to AgentCore Runtime. It uses Strands Agents SDK with Lambda-backed tools.

In [ ]:
%%writefile certagent_server.py
import os
import json
import boto3
from strands import Agent, tool
from strands.models.bedrock import BedrockModel
from http.server import HTTPServer, BaseHTTPRequestHandler

AWS_REGION = os.environ.get('AWS_REGION', 'us-east-1')
lambda_client = boto3.client('lambda', region_name=AWS_REGION)

LAMBDA_SCAN = os.environ.get('LAMBDA_SCAN', 'certagent-scan-certificates')
LAMBDA_RENEW = os.environ.get('LAMBDA_RENEW', 'certagent-renew-certificate')
LAMBDA_INVENTORY = os.environ.get('LAMBDA_INVENTORY', 'certagent-list-inventory')

def invoke_lambda(fn_name, payload):
    r = lambda_client.invoke(FunctionName=fn_name, InvocationType='RequestResponse',
                             Payload=json.dumps(payload))
    raw = json.loads(r['Payload'].read())
    return raw.get('body', raw)

@tool
def scan_certificates(threshold_days: int = 30, use_mock: bool = True) -> str:
    """Scan for certificates expiring within the given threshold days.
    Args:
        threshold_days: Number of days to look ahead.
        use_mock: Use mock data instead of real DigiCert API.
    """
    result = invoke_lambda(LAMBDA_SCAN, {'threshold_days': threshold_days, 'use_mock': use_mock})
    return json.dumps(result, indent=2, default=str)

@tool
def renew_certificate(order_id: str, common_name: str, use_mock: bool = True) -> str:
    """Renew a certificate by order ID and domain name.
    Args:
        order_id: The DigiCert order ID to renew.
        common_name: The domain name of the certificate.
        use_mock: Use mock mode.
    """
    result = invoke_lambda(LAMBDA_RENEW, {
        'order_id': order_id, 'common_name': common_name,
        'sans': [common_name], 'use_mock': use_mock
    })
    return json.dumps(result, indent=2, default=str)

@tool
def list_inventory(status: str = 'all') -> str:
    """List the certificate inventory.
    Args:
        status: Filter: all, pending, submitted, issued, completed.
    """
    result = invoke_lambda(LAMBDA_INVENTORY, {'status': status})
    return json.dumps(result, indent=2, default=str)

SYSTEM_PROMPT = '''You are CertAgent, an AI operations agent for TLS/SSL certificate lifecycle.
Capabilities: SCAN, RENEW, INVENTORY.
Rules:
- Always use mock mode (use_mock=true)
- Report priority: EXPIRED, CRITICAL (<7d), HIGH (<14d), MEDIUM (<30d), LOW
- Be concise. After any action, state what was done and what happens next.'''

model = BedrockModel(model_id='anthropic.claude-sonnet-4-20250514-v1:0', region_name=AWS_REGION)
agent = Agent(model=model, system_prompt=SYSTEM_PROMPT,
              tools=[scan_certificates, renew_certificate, list_inventory])


class AgentHandler(BaseHTTPRequestHandler):
    def do_POST(self):
        content_length = int(self.headers.get('Content-Length', 0))
        body = self.rfile.read(content_length)
        try:
            event = json.loads(body) if body else {}
            input_text = event.get('inputText', event.get('prompt', ''))
            if not input_text:
                result = {'output': 'Please provide a message.'}
            else:
                response = agent(input_text)
                result = {'output': str(response)}
            self.send_response(200)
            self.send_header('Content-Type', 'application/json')
            self.end_headers()
            self.wfile.write(json.dumps(result).encode())
        except Exception as e:
            self.send_response(500)
            self.send_header('Content-Type', 'application/json')
            self.end_headers()
            self.wfile.write(json.dumps({'error': str(e)}).encode())

    def do_GET(self):
        self.send_response(200)
        self.send_header('Content-Type', 'application/json')
        self.end_headers()
        self.wfile.write(json.dumps({'status': 'healthy', 'agent': 'CertAgent'}).encode())


if __name__ == '__main__':
    port = int(os.environ.get('PORT', '8080'))
    server = HTTPServer(('0.0.0.0', port), AgentHandler)
    print(f'CertAgent HTTP server starting on port {port}')
    server.serve_forever()

## Step 4 - Write requirements file

In [ ]:
%%writefile certagent_requirements.txt
strands-agents
strands-agents-bedrock
boto3>=1.35.0

## Step 5 - Clean any stale config from previous runs

In [ ]:
import shutil, glob
for f in glob.glob('.bedrock_agentcore*'):
    if os.path.isfile(f): os.remove(f)
    elif os.path.isdir(f): shutil.rmtree(f)
print('Stale config cleared (safe to run on first attempt too)')

## Step 6 - Configure AgentCore Runtime

This generates the Dockerfile and sets up the deployment configuration.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime = Runtime()

print('Configuring AgentCore Runtime...')
agentcore_runtime.configure(
    entrypoint='certagent_server.py',
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file='certagent_requirements.txt',
    region=region,
    protocol='HTTP',
    agent_name='certagent',
)
print('Configuration completed')

## Step 7 - Launch to AgentCore Runtime

This builds the container via CodeBuild (ARM64), pushes to ECR, and deploys.

**Takes 3-5 minutes.** Watch for the CodeBuild phase progress.

In [ ]:
print('Launching CertAgent to AgentCore Runtime...')
print('This takes 3-5 minutes...')
launch_result = agentcore_runtime.launch()
print(f'\nAgent ARN: {launch_result.agent_arn}')
print(f'Agent ID: {launch_result.agent_id}')
print('\nDeployment successful!')

## Step 8 - Invoke the deployed agent

Now invoke the agent running in AgentCore Runtime.

In [ ]:
print('Invoking CertAgent...')
response = agentcore_runtime.invoke('What certificates are expiring soon? Use mock mode and give a prioritized summary.')
print(response)

In [ ]:
print('Asking to renew...')
response = agentcore_runtime.invoke('Renew the certificate for api.example.com using mock mode')
print(response)

In [ ]:
print('Multi-step operation...')
response = agentcore_runtime.invoke('Scan for certs expiring in 7 days with mock data, then renew any CRITICAL ones')
print(response)

In [ ]:
print('Inventory...')
response = agentcore_runtime.invoke('Show the full certificate inventory')
print(response)

## Step 9 - Check agent status

In [ ]:
status = agentcore_runtime.status()
print(f'Agent status: {status}')

## Troubleshooting

If invoke fails with `ResourceNotFoundException`:
1. The agent may still be initializing. Wait 60 seconds and retry.
2. Run `agentcore_runtime.status()` to check if the endpoint is ready.

If you get `ValueError: Must configure and launch first`:
- Do NOT restart the kernel between configure/launch/invoke.
- If kernel was restarted, re-run from Step 5 (clean config) onwards.

To invoke via boto3 directly (if the Runtime object is lost):
```python
import boto3, json
client = boto3.client('bedrock-agentcore', region_name=region)
response = client.invoke_agent_runtime(
    agentRuntimeArn='YOUR_AGENT_ARN',
    qualifier='DEFAULT',
    payload=json.dumps({'prompt': 'your question'}).encode()
)
print(response['body'].read().decode())
```

## Lab 03 Complete

You have:
1. Written a Strands-based agent with Lambda-backed tools
2. Configured the AgentCore Runtime deployment (Docker, ECR, IAM)
3. Deployed the agent via CodeBuild to AgentCore Runtime
4. Invoked the persistent agent with multiple operations

The agent is now running as a managed service.

**Next:** `04_proactive_monitoring.ipynb`